In [3]:
from pathlib import Path
from math import dist

import numpy as np
import py3Dmol

In [4]:
# The notebook is inside the Analysis folder.
# Therefore, the project root is one folder above the notebook.
PROJECT_ROOT = Path.cwd().parent

XYZ_DIR = PROJECT_ROOT / "xyz_files"

reactant_path = XYZ_DIR / "BHPERI_ortho-xylylene.xyz"
ts_path = XYZ_DIR / "BHPERI_TS3.xyz"

print("Project root:", PROJECT_ROOT)
print("Reactant file:", reactant_path)
print("Transition-state file:", ts_path)

print("\nFile checks:")
print("Reactant exists:", reactant_path.exists())
print("TS exists:", ts_path.exists())

Project root: c:\Users\91988\GSCDB Benchmarking
Reactant file: c:\Users\91988\GSCDB Benchmarking\xyz_files\BHPERI_ortho-xylylene.xyz
Transition-state file: c:\Users\91988\GSCDB Benchmarking\xyz_files\BHPERI_TS3.xyz

File checks:
Reactant exists: True
TS exists: True


In [5]:
def read_xyz_file(file_path):
    """
    Read an XYZ file.

    Returns
    -------
    xyz_text : str
        Complete XYZ file content, suitable for py3Dmol.
    atoms : list[dict]
        Element symbols and Cartesian coordinates.
    comment : str
        Second line of the XYZ file.
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"XYZ file not found: {file_path}")

    lines = file_path.read_text(encoding="utf-8").splitlines()

    if len(lines) < 3:
        raise ValueError(f"{file_path.name} is not a valid XYZ file.")

    number_of_atoms = int(lines[0].strip())
    comment = lines[1].strip()

    coordinate_lines = lines[2 : 2 + number_of_atoms]

    if len(coordinate_lines) != number_of_atoms:
        raise ValueError(
            f"{file_path.name}: expected {number_of_atoms} atoms, "
            f"but found {len(coordinate_lines)} coordinate lines."
        )

    atoms = []

    for atom_number, line in enumerate(coordinate_lines, start=1):
        parts = line.split()

        if len(parts) < 4:
            raise ValueError(
                f"Invalid coordinate line for atom {atom_number}: {line}"
            )

        element = parts[0]
        x, y, z = map(float, parts[1:4])

        atoms.append(
            {
                "atom_number": atom_number,
                "element": element,
                "x": x,
                "y": y,
                "z": z,
            }
        )

    xyz_text = file_path.read_text(encoding="utf-8")

    return xyz_text, atoms, comment

In [6]:
reactant_xyz, reactant_atoms, reactant_comment = read_xyz_file(
    reactant_path
)

ts_xyz, ts_atoms, ts_comment = read_xyz_file(
    ts_path
)

print("Reactant atoms:", len(reactant_atoms))
print("TS atoms:", len(ts_atoms))

print("\nReactant metadata:")
print(reactant_comment)

print("\nTransition-state metadata:")
print(ts_comment)

Reactant atoms: 16
TS atoms: 16

Reactant metadata:
charge=0, multiplicity=1, basis=def2-QZVPPD, mem_total=7500, AUX_BASIS_CORR=rimp2-def2-QZVPPD, SCF_ALGORITHM=GDM, xc_grid=000099000590, num_basis=768, num_pairs=265191, num_threads=2

Transition-state metadata:
charge=0, multiplicity=1, basis=def2-QZVPPD, mem_total=7500, AUX_BASIS_CORR=rimp2-def2-QZVPPD, SCF_ALGORITHM=GDM, xc_grid=000099000590, num_basis=768, num_pairs=268012, num_threads=2


In [7]:
reactant_view = py3Dmol.view(width=600, height=450)

reactant_view.addModel(reactant_xyz, "xyz")

reactant_view.setStyle(
    {},
    {
        "stick": {
            "radius": 0.15
        },
        "sphere": {
            "scale": 0.28
        }
    }
)

reactant_view.setBackgroundColor("white")
reactant_view.zoomTo()

reactant_view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [8]:
ts_view = py3Dmol.view(width=600, height=450)

ts_view.addModel(ts_xyz, "xyz")

ts_view.setStyle(
    {},
    {
        "stick": {
            "radius": 0.15
        },
        "sphere": {
            "scale": 0.28
        }
    }
)

ts_view.setBackgroundColor("white")
ts_view.zoomTo()

ts_view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
viewer = py3Dmol.view(
    width=1100,
    height=480,
    viewergrid=(1, 2),
    linked=False
)

# Left panel: reactant
viewer.addModel(
    reactant_xyz,
    "xyz",
    viewer=(0, 0)
)

viewer.setStyle(
    {},
    {
        "stick": {"radius": 0.15},
        "sphere": {"scale": 0.28}
    },
    viewer=(0, 0)
)

viewer.setBackgroundColor(
    "white",
    viewer=(0, 0)
)

viewer.zoomTo(
    viewer=(0, 0)
)

# Right panel: transition state
viewer.addModel(
    ts_xyz,
    "xyz",
    viewer=(0, 1)
)

viewer.setStyle(
    {},
    {
        "stick": {"radius": 0.15},
        "sphere": {"scale": 0.28}
    },
    viewer=(0, 1)
)

viewer.setBackgroundColor(
    "white",
    viewer=(0, 1)
)

viewer.zoomTo(
    viewer=(0, 1)
)

viewer

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [11]:
def atom_position(atoms, atom_number):
    """
    Return an atom's XYZ position.

    atom_number is one-based, matching the XYZ-file row numbering.
    """
    atom = atoms[atom_number - 1]

    return np.array(
        [atom["x"], atom["y"], atom["z"]],
        dtype=float
    )


def distance_between_atoms(atoms, atom_1, atom_2):
    position_1 = atom_position(atoms, atom_1)
    position_2 = atom_position(atoms, atom_2)

    return np.linalg.norm(position_1 - position_2)

reactant_terminal_distance = distance_between_atoms(
    reactant_atoms,
    11,
    12
)

ts_terminal_distance = distance_between_atoms(
    ts_atoms,
    8,
    16
)

distance_change = (
    ts_terminal_distance
    - reactant_terminal_distance
)

print(
    f"Reactant C11···C12 distance: "
    f"{reactant_terminal_distance:.3f} Å"
)

print(
    f"TS C8···C16 distance: "
    f"{ts_terminal_distance:.3f} Å"
)

print(
    f"Distance change: "
    f"{distance_change:.3f} Å"
)

Reactant C11···C12 distance: 3.006 Å
TS C8···C16 distance: 2.292 Å
Distance change: -0.714 Å


In [12]:
def coordinate_dictionary(atoms, atom_number):
    atom = atoms[atom_number - 1]

    return {
        "x": atom["x"],
        "y": atom["y"],
        "z": atom["z"],
    }


def midpoint_dictionary(atoms, atom_1, atom_2):
    point_1 = atom_position(atoms, atom_1)
    point_2 = atom_position(atoms, atom_2)

    midpoint = (point_1 + point_2) / 2.0

    return {
        "x": float(midpoint[0]),
        "y": float(midpoint[1]),
        "z": float(midpoint[2]),
    }

In [13]:
viewer = py3Dmol.view(
    width=1100,
    height=500,
    viewergrid=(1, 2),
    linked=False
)

# -------------------------------------------------
# Left panel: reactant
# -------------------------------------------------

viewer.addModel(
    reactant_xyz,
    "xyz",
    viewer=(0, 0)
)

viewer.setStyle(
    {},
    {
        "stick": {"radius": 0.14},
        "sphere": {"scale": 0.27}
    },
    viewer=(0, 0)
)

reactant_c1 = coordinate_dictionary(
    reactant_atoms,
    11
)

reactant_c2 = coordinate_dictionary(
    reactant_atoms,
    12
)

viewer.addCylinder(
    {
        "start": reactant_c1,
        "end": reactant_c2,
        "radius": 0.045,
        "color": "grey",
        "opacity": 0.65,
        "fromCap": True,
        "toCap": True,
    },
    viewer=(0, 0)
)

viewer.addLabel(
    f"C···C = {reactant_terminal_distance:.3f} Å",
    {
        "position": midpoint_dictionary(
            reactant_atoms,
            11,
            12
        ),
        "backgroundColor": "white",
        "fontColor": "black",
        "fontSize": 15,
        "showBackground": True,
    },
    viewer=(0, 0)
)

viewer.addLabel(
    "BHPERI_ortho-xylylene",
    {
        "position": {
            "x": 0,
            "y": 0,
            "z": 3.8
        },
        "backgroundColor": "white",
        "fontColor": "black",
        "fontSize": 17,
        "showBackground": True,
    },
    viewer=(0, 0)
)

viewer.setBackgroundColor(
    "white",
    viewer=(0, 0)
)

viewer.zoomTo(
    viewer=(0, 0)
)

# -------------------------------------------------
# Right panel: transition state
# -------------------------------------------------

viewer.addModel(
    ts_xyz,
    "xyz",
    viewer=(0, 1)
)

viewer.setStyle(
    {},
    {
        "stick": {"radius": 0.14},
        "sphere": {"scale": 0.27}
    },
    viewer=(0, 1)
)

ts_c1 = coordinate_dictionary(
    ts_atoms,
    8
)

ts_c2 = coordinate_dictionary(
    ts_atoms,
    16
)

viewer.addCylinder(
    {
        "start": ts_c1,
        "end": ts_c2,
        "radius": 0.055,
        "color": "red",
        "opacity": 0.85,
        "fromCap": True,
        "toCap": True,
    },
    viewer=(0, 1)
)

viewer.addLabel(
    f"C···C = {ts_terminal_distance:.3f} Å",
    {
        "position": midpoint_dictionary(
            ts_atoms,
            8,
            16
        ),
        "backgroundColor": "white",
        "fontColor": "red",
        "fontSize": 15,
        "showBackground": True,
    },
    viewer=(0, 1)
)

viewer.addLabel(
    "BHPERI_TS3",
    {
        "position": {
            "x": 0,
            "y": 0,
            "z": 3.8
        },
        "backgroundColor": "white",
        "fontColor": "black",
        "fontSize": 17,
        "showBackground": True,
    },
    viewer=(0, 1)
)

viewer.setBackgroundColor(
    "white",
    viewer=(0, 1)
)

viewer.zoomTo(
    viewer=(0, 1)
)

viewer

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [14]:
# Reactant atom labels
viewer.addLabel(
    "C11",
    {
        "position": coordinate_dictionary(
            reactant_atoms,
            11
        ),
        "fontColor": "blue",
        "backgroundColor": "white",
        "fontSize": 14,
    },
    viewer=(0, 0)
)

viewer.addLabel(
    "C12",
    {
        "position": coordinate_dictionary(
            reactant_atoms,
            12
        ),
        "fontColor": "blue",
        "backgroundColor": "white",
        "fontSize": 14,
    },
    viewer=(0, 0)
)

# Transition-state atom labels
viewer.addLabel(
    "C8",
    {
        "position": coordinate_dictionary(
            ts_atoms,
            8
        ),
        "fontColor": "red",
        "backgroundColor": "white",
        "fontSize": 14,
    },
    viewer=(0, 1)
)

viewer.addLabel(
    "C16",
    {
        "position": coordinate_dictionary(
            ts_atoms,
            16
        ),
        "fontColor": "red",
        "backgroundColor": "white",
        "fontSize": 14,
    },
    viewer=(0, 1)
)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [15]:
def print_atom_table(atoms, structure_name):
    print(f"\n{structure_name}")
    print("-" * 62)
    print(
        f"{'Atom':>5} "
        f"{'Element':>8} "
        f"{'X / Å':>12} "
        f"{'Y / Å':>12} "
        f"{'Z / Å':>12}"
    )
    print("-" * 62)

    for atom in atoms:
        print(
            f"{atom['atom_number']:>5} "
            f"{atom['element']:>8} "
            f"{atom['x']:>12.6f} "
            f"{atom['y']:>12.6f} "
            f"{atom['z']:>12.6f}"
        )


print_atom_table(
    reactant_atoms,
    "BHPERI_ortho-xylylene"
)

print_atom_table(
    ts_atoms,
    "BHPERI_TS3"
)


BHPERI_ortho-xylylene
--------------------------------------------------------------
 Atom  Element        X / Å        Y / Å        Z / Å
--------------------------------------------------------------
    1        H    -0.144883     1.244000     2.790788
    2        C    -0.079370     0.721429     1.840230
    3        C    -0.128693     1.408986     0.677482
    4        C     0.079370    -0.721429     1.840230
    5        C     0.128693    -1.408986     0.677482
    6        H     0.144883    -1.244000     2.790788
    7        H    -0.221218     2.492346     0.677508
    8        C    -0.006727     0.748467    -0.621344
    9        C     0.006727    -0.748467    -0.621344
   10        H     0.221218    -2.492346     0.677508
   11        C     0.128693     1.497448    -1.739510
   12        C    -0.128693    -1.497448    -1.739510
   13        H     0.089546     2.581731    -1.689011
   14        H     0.294457     1.064480    -2.720428
   15        H    -0.294457    -1.064480 